# Ablation Studies Notebook

This notebook focuses on epsilon ablation across retrievers with a single generation model.


## 0) Environment

Run with your project venv:

```bash
cd /Users/shayan/Projects/PAS_RAG
source .venv_pas_rag/bin/activate
```


In [1]:
from pathlib import Path
from collections import Counter
import csv
import json
import math
import re
import statistics
import time
import os

from utils import generate_grounded_answer
import pas_rag_end_to_end_patched as pas

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

BASE = Path.cwd()
OUTPUT_DIR = BASE / 'ablation_results'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Working directory:', BASE)
print('Output directory:', OUTPUT_DIR)



Working directory: /Users/shayan/Projects/PAS_RAG
Output directory: /Users/shayan/Projects/PAS_RAG/ablation_results


/Users/shayan/Projects/PAS_RAG/.venv_pas_rag/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1) Load Data


In [2]:
anchors = pas.load_jsonl(BASE / 'expanded_anchor_registry.jsonl')
chunks = pas.load_jsonl(BASE / 'expanded_chunks.jsonl')
queries = pas.load_jsonl(BASE / 'expanded_eval_queries.jsonl')

print('anchors:', len(anchors))
print('chunks:', len(chunks))
print('queries:', len(queries))



anchors: 30
chunks: 1010
queries: 423


## 2) Config

- `top_k`: retrieval cutoff
- `max_queries`: set an int (e.g., 50) for quick smoke runs
- `epsilon`: PAS privacy parameter
- `lambda_weight`: semantic/spatial fusion weight in retrieval score


In [3]:
RETRIEVER_OPTIONS = {
    'minilm': 'sentence-transformers/all-MiniLM-L6-v2',
    'bge-base': 'BAAI/bge-base-en-v1.5',
}

# Choose retriever here: 'minilm' (original) or 'bge-base' (second retriever)
RETRIEVER_CHOICE = 'bge-base'

CONFIG = {
    'top_k': 5,
    'max_queries': None,
    'mc_samples': 1000,
    'epsilon': 1.0,
    'scale_m': 500.0,
    'lambda_weight': 0.8,
    'progress_every': 100,
    'model_name': 'gemma-3-12b-it',
    'retriever': RETRIEVER_CHOICE,
    'embed_device': 'mps',  # on Mac, use Metal GPU if available
}


def apply_retriever_choice(cfg):
    key = cfg.get('retriever', 'minilm')
    if key not in RETRIEVER_OPTIONS:
        raise ValueError(f"Unknown retriever '{key}'. Valid: {list(RETRIEVER_OPTIONS)}")
    cfg['pas_embed_model'] = RETRIEVER_OPTIONS[key]
    os.environ['PAS_EMBED_MODEL'] = cfg['pas_embed_model']
    os.environ['PAS_EMBED_DEVICE'] = cfg.get('embed_device', 'mps')


apply_retriever_choice(CONFIG)

print(CONFIG)
print('PAS_EMBED_MODEL=', os.environ.get('PAS_EMBED_MODEL'))
print('PAS_EMBED_DEVICE=', os.environ.get('PAS_EMBED_DEVICE'))



{'top_k': 5, 'max_queries': None, 'mc_samples': 1000, 'epsilon': 1.0, 'scale_m': 500.0, 'lambda_weight': 0.8, 'progress_every': 100, 'model_name': 'gemma-3-12b-it', 'retriever': 'bge-base', 'embed_device': 'mps', 'pas_embed_model': 'BAAI/bge-base-en-v1.5'}
PAS_EMBED_MODEL= BAAI/bge-base-en-v1.5
PAS_EMBED_DEVICE= mps


## 3) Metric Helpers

Assumption for `ALE`: in this notebook, `ALE` is modeled as **attacker localization error** in meters,
using a simple token-centroid attacker estimate derived from released PAS `(anchor, direction_bin, distance_bin)`.


In [4]:
def normalize_token(t):
    return re.sub(r'[^a-z0-9]+', '', t.lower())


def tokenize_text(s):
    return [normalize_token(x) for x in re.findall(r"[A-Za-z0-9']+", s.lower()) if normalize_token(x)]


def token_f1(pred, ref):
    p = tokenize_text(pred)
    r = tokenize_text(ref)
    if not p or not r:
        return 0.0
    pc = Counter(p)
    rc = Counter(r)
    overlap = sum((pc & rc).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(p)
    recall = overlap / len(r)
    return 2 * precision * recall / (precision + recall)


def dcg_at_k(relevances, k):
    rel = relevances[:k]
    return sum((r / math.log2(i + 2)) for i, r in enumerate(rel))


def ndcg_at_k(retrieved_doc_ids, ground_truth_doc_ids, k):
    gt = set(ground_truth_doc_ids)
    rel = [1 if d in gt else 0 for d in retrieved_doc_ids[:k]]
    dcg = dcg_at_k(rel, k)
    ideal_rel = [1] * min(len(gt), k)
    idcg = dcg_at_k(ideal_rel, k)
    return (dcg / idcg) if idcg > 0 else 0.0


def citation_correctness(citations, ground_truth_doc_ids):
    gt = set(ground_truth_doc_ids)
    if not citations:
        return 0.0
    cited = [c.get('doc_id') for c in citations if c.get('doc_id')]
    if not cited:
        return 0.0
    correct = sum(1 for d in cited if d in gt)
    return correct / len(cited)


def ring_bounds_m(distance_bin):
    # Support both original bins and newer narrower bins.
    if distance_bin == '0-0.25mi':
        return 0.0, 0.25 * 1609.344
    if distance_bin == '0.25-0.5mi':
        return 0.25 * 1609.344, 0.5 * 1609.344
    if distance_bin == '0.5-0.75mi':
        return 0.5 * 1609.344, 0.75 * 1609.344
    if distance_bin == '0.75-1mi':
        return 0.75 * 1609.344, 1.0 * 1609.344
    if distance_bin == '1-1.5mi':
        return 1.0 * 1609.344, 1.5 * 1609.344
    if distance_bin == '1.5-2mi':
        return 1.5 * 1609.344, 2.0 * 1609.344

    if distance_bin == '0-0.5mi':
        return 0.0, 0.5 * 1609.344
    if distance_bin == '0.5-1mi':
        return 0.5 * 1609.344, 1.0 * 1609.344
    if distance_bin == '1-2mi':
        return 1.0 * 1609.344, 2.0 * 1609.344

    # Covers 2mi+/2+mi and unknowns conservatively.
    return 2.0 * 1609.344, 3.0 * 1609.344


def ale_from_token(query, token, anchor_lookup):
    # Attacker estimate: sector midpoint of released PAS token.
    anchor = anchor_lookup[token['anchor_id']]
    anchor_loc = (anchor['geo']['lat'], anchor['geo']['lon'])
    rmin, rmax = ring_bounds_m(token['distance_bin'])
    r_mid = (rmin + rmax) / 2.0

    center_map = {
        # 8-bin
        'N': 0, 'NE': 45, 'E': 90, 'SE': 135, 'S': 180, 'SW': 225, 'W': 270, 'NW': 315,
        # 16-bin
        'NNE': 22.5, 'ENE': 67.5, 'ESE': 112.5, 'SSE': 157.5,
        'SSW': 202.5, 'WSW': 247.5, 'WNW': 292.5, 'NNW': 337.5,
        'ANY': 0,
    }
    b = center_map.get(token['direction_bin'], 0)
    est = pas.offset_from_anchor(anchor_loc[0], anchor_loc[1], r_mid, b)

    true_loc = (
        query['spatial_intent']['user_location']['lat'],
        query['spatial_intent']['user_location']['lon'],
    )
    return pas.haversine_m(true_loc, est)


def build_reference_answer_from_ground_truth(query, gt_chunks):
    if not gt_chunks:
        return ''
    titles = ', '.join(c['title'] for c in gt_chunks[:3])
    facts = []
    for c in gt_chunks[:3]:
        facts.extend(c.get('supporting_facts', [])[:1])
    return f"For query '{query['raw_query']}', relevant places include {titles}. " + ' '.join(facts)


## 4) Retrieval Helpers


In [5]:
chunk_by_doc = {}
for c in chunks:
    # Keep first chunk per doc_id in this dataset layout.
    chunk_by_doc.setdefault(c['doc_id'], c)


def exact_location_retrieve(query, chunks, top_k=3, lambda_weight=0.65):
    user_loc = (
        query['spatial_intent']['user_location']['lat'],
        query['spatial_intent']['user_location']['lon'],
    )
    radius_m = query['spatial_intent']['radius_miles'] * 1609.344
    required_dir = query['spatial_intent']['direction_constraint']

    ranked = []
    for chunk in chunks:
        sem = pas.semantic_score(query, chunk)
        geo = chunk.get('metadata', {}).get('geo', {}) or {}
        lat, lon = geo.get('lat'), geo.get('lon')
        spa = 0.0

        if lat is not None and lon is not None:
            item_loc = (lat, lon)
            d = pas.haversine_m(user_loc, item_loc)
            within = d <= radius_m
            directional = True if required_dir == 'ANY' else (pas.dir_bin(pas.bearing_deg(user_loc, item_loc)) == required_dir)
            if within and directional:
                spa = 1.0
            elif within:
                spa = 0.45

        score = lambda_weight * sem + (1 - lambda_weight) * spa
        ranked.append({
            'doc_id': chunk['doc_id'],
            'chunk_id': chunk['chunk_id'],
            'title': chunk['title'],
            'category': chunk['category'],
            'score': round(score, 4),
            'semantic_score': round(sem, 4),
            'spatial_score': round(spa, 4),
            'metadata': chunk['metadata'],
            'content': chunk['content'],
            'supporting_facts': chunk.get('supporting_facts', []),
        })

    ranked.sort(key=lambda x: (x['score'], x['semantic_score'], x['spatial_score'], x['title']), reverse=True)
    return ranked[:top_k]


## 6) Defense Evaluation Cell (PAS)

This cell evaluates **PAS defense only** for a fixed `(epsilon, lambda)`.


In [6]:
def evaluate_pas_defense(queries, chunks, anchors, cfg, show_progress=True):
    anchor_lookup = {a['anchor_id']: a for a in anchors}
    
    selected = queries[:cfg['max_queries']] if cfg['max_queries'] else queries

    debug_n = int(cfg.get('debug_failures_n', 5))
    debug_f1_threshold = float(cfg.get('debug_f1_threshold', 0.4))
    failure_cases = []

    recalls, ndcgs, f1s, cites, ales = [], [], [], [], []
    rows = []

    iterator = tqdm(selected, total=len(selected), desc='PAS defense eval', unit='query') if show_progress else selected
    for i, q in enumerate(iterator, start=1):
        parsed = pas.parse_query(q['raw_query'], fallback_query=q)
        token = pas.build_pas_token(parsed, anchors, epsilon=cfg['epsilon'], scale_m=cfg['scale_m'])
        latent = pas.build_latent_user_samples(parsed, token, mc_samples=cfg['mc_samples'])
        top = pas.hybrid_retrieve(
            parsed,
            chunks,
            pas_token=token,
            latent_user_samples=latent,
            top_k=cfg['top_k'],
            lambda_weight=cfg['lambda_weight'],
        )
        top_doc_ids = [r['doc_id'] for r in top]

        rec = pas.recall_at_k(q['ground_truth_doc_ids'], top, k=cfg['top_k'])
        nd = ndcg_at_k(top_doc_ids, q['ground_truth_doc_ids'], cfg['top_k'])

        gen = generate_grounded_answer(q, top, model_name=cfg['model_name'])
        pred_answer = gen['answer']
        
        gt_chunks = [chunk_by_doc[d] for d in q['ground_truth_doc_ids'] if d in chunk_by_doc]
        ref_answer = build_reference_answer_from_ground_truth(q, gt_chunks)

        f1 = token_f1(pred_answer, ref_answer)
        cc = citation_correctness(gen.get('citations', []), q['ground_truth_doc_ids'])
        ale = ale_from_token(q, token, anchor_lookup)

        if f1 <= debug_f1_threshold:
            failure_cases.append({
                'qid': q.get('qid'),
                'query': q.get('raw_query', ''),
                'pred_answer': pred_answer,
                'ref_answer': ref_answer,
                'f1': f1,
                'recall': rec,
                'ndcg': nd,
                'citation_correctness': cc,
                'ale_m': ale,
                'top_doc_ids': top_doc_ids,
                'ground_truth_doc_ids': q.get('ground_truth_doc_ids', []),
            })

        recalls.append(rec)
        ndcgs.append(nd)
        f1s.append(f1)
        cites.append(cc)
        ales.append(ale)

        rows.append({
            'qid': q['qid'],
            'query': q['raw_query'],
            'mode': 'pas_defense',
            'model_name': cfg['model_name'],
            'retriever': cfg['retriever'],
            'epsilon': cfg['epsilon'],
            'lambda_weight': cfg['lambda_weight'],
            'recall_at_k': round(rec, 4),
            'ndcg_at_k': round(nd, 4),
            'gen_f1': round(f1, 4),
            'citation_correctness': round(cc, 4),
            'ale_m': round(ale, 2),
            'pas_anchor_id': token['anchor_id'],
            'pas_direction_bin': token['direction_bin'],
            'pas_distance_bin': token['distance_bin'],
            'top_doc_ids': ';'.join(top_doc_ids),
        })

        if show_progress and i % cfg['progress_every'] == 0 and hasattr(iterator, 'set_postfix'):
            iterator.set_postfix(recall=round(statistics.mean(recalls), 3), ndcg=round(statistics.mean(ndcgs), 3), ale=round(statistics.mean(ales), 1))

    # Print a compact sample of poor-performing examples for error analysis.
    if failure_cases and debug_n > 0:
        failure_cases_sorted = sorted(failure_cases, key=lambda x: x['f1'])[:debug_n]
        print(f"\n=== PAS DEBUG: showing {len(failure_cases_sorted)} low-F1 cases (threshold <= {debug_f1_threshold}) ===")
        for k, ex in enumerate(failure_cases_sorted, start=1):
            print(f"\n--- Failure Case {k} | qid={ex['qid']} | f1={ex['f1']:.4f} | recall={ex['recall']:.4f} | ndcg={ex['ndcg']:.4f} | ale_m={ex['ale_m']:.2f} ---")
            print('Query:')
            print(ex['query'])
            print('Model response:')
            print(ex['pred_answer'])
            print('Ground truth (reference answer used for F1):')
            print(ex['ref_answer'])
            print('Top doc ids:')
            print(ex['top_doc_ids'])
            print('Ground-truth doc ids:')
            print(ex['ground_truth_doc_ids'])

    summary = {
        'mode': 'pas_defense',
        'model_name': cfg['model_name'],
        'retriever': cfg['retriever'],
            'retriever': cfg['retriever'],
        'n_queries': len(selected),
        'epsilon': cfg['epsilon'],
        'lambda_weight': cfg['lambda_weight'],
        'mean_recall_at_k': round(statistics.mean(recalls), 4),
        'mean_ndcg_at_k': round(statistics.mean(ndcgs), 4),
        'mean_gen_f1': round(statistics.mean(f1s), 4),
        'mean_citation_correctness': round(statistics.mean(cites), 4),
        'mean_ale_m': round(statistics.mean(ales), 2),
    }
    return summary, rows

# print(CONFIG)
# pas_summary, pas_rows = evaluate_pas_defense(queries, chunks, anchors, CONFIG)


## Epsilon Ablation (Retrievers)

Run PAS defense with epsilon values `[10, 20, 50, 100]` for each retriever (`minilm`, `bge-base`) using one generation model.


In [7]:
from copy import deepcopy
import pandas as pd

ABLATION_EPSILONS = [10.0, 20.0, 50.0, 100.0]
ABLATION_LAMBDAS = [0.7, 0.8]
ABLATION_RETRIEVERS = ['minilm', 'bge-base']
# Keep a single generation model across all ablation runs
ABLATION_MODEL = CONFIG['model_name']
ABLATION_MAX_QUERIES = 250

queries_subset = queries[:ABLATION_MAX_QUERIES]

print('Ablation model:', ABLATION_MODEL)
print('Ablation retrievers:', ABLATION_RETRIEVERS)
print('Ablation epsilons:', ABLATION_EPSILONS)
print('Ablation lambdas:', ABLATION_LAMBDAS)
print('Ablation query count:', len(queries_subset), 'of', len(queries))


def _fmt_num(x):
    x = float(x)
    return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')


ablation_rows = []
ablation_details = {}

for retriever in ABLATION_RETRIEVERS:
    base_cfg = deepcopy(CONFIG)
    base_cfg['model_name'] = ABLATION_MODEL
    base_cfg['retriever'] = retriever
    apply_retriever_choice(base_cfg)

    for eps in ABLATION_EPSILONS:
        for lam in ABLATION_LAMBDAS:
            eps_s = _fmt_num(eps)
            lam_s = _fmt_num(lam)
            per_run_name = f"ablation_results_{retriever}_epsilon_{eps_s}_lambda_{lam_s}.json"
            per_run_path = OUTPUT_DIR / per_run_name

            if per_run_path.exists():
                print('Skipping existing:', per_run_path)
                try:
                    existing = json.loads(per_run_path.read_text())
                    p_summary = existing.get('pas_summary', {})
                    row = {
                        'model_name': existing.get('config', {}).get('model_name', ABLATION_MODEL),
                        'retriever': existing.get('config', {}).get('retriever', retriever),
                        'epsilon': existing.get('config', {}).get('epsilon', eps),
                        'lambda_weight': existing.get('config', {}).get('lambda_weight', lam),
                        'pas_mean_recall_at_k': p_summary.get('mean_recall_at_k'),
                        'pas_mean_ndcg_at_k': p_summary.get('mean_ndcg_at_k'),
                        'pas_mean_token_f1': p_summary.get('mean_token_f1'),
                        'pas_mean_citation_correctness': p_summary.get('mean_citation_correctness'),
                        'pas_mean_ale_m': p_summary.get('mean_ale_m'),
                    }
                    ablation_rows.append(row)
                    tag = f"{row['model_name'].replace('/', '-') }__{row['retriever']}__eps{eps_s}__lam{lam_s}"
                    ablation_details[tag] = existing
                except Exception as e:
                    print('Warning: could not parse existing file, leaving out of DataFrame:', e)
                continue

            cfg = deepcopy(base_cfg)
            cfg['epsilon'] = eps
            cfg['lambda_weight'] = lam

            p_summary, p_rows = evaluate_pas_defense(queries_subset, chunks, anchors, cfg)

            row = {
                'model_name': ABLATION_MODEL,
                'retriever': retriever,
                'epsilon': eps,
                'lambda_weight': cfg['lambda_weight'],
                'pas_mean_recall_at_k': p_summary.get('mean_recall_at_k'),
                'pas_mean_ndcg_at_k': p_summary.get('mean_ndcg_at_k'),
                'pas_mean_token_f1': p_summary.get('mean_token_f1'),
                'pas_mean_citation_correctness': p_summary.get('mean_citation_correctness'),
                'pas_mean_ale_m': p_summary.get('mean_ale_m'),
            }
            ablation_rows.append(row)

            tag = f"{ABLATION_MODEL.replace('/', '-') }__{retriever}__eps{eps_s}__lam{lam_s}"
            detail = {
                'config': {
                    'model_name': ABLATION_MODEL,
                    'retriever': retriever,
                    'epsilon': eps,
                    'lambda_weight': lam,
                    'max_queries': ABLATION_MAX_QUERIES,
                },
                'pas_summary': p_summary,
                'pas_rows': p_rows,
            }
            ablation_details[tag] = detail

            # checkpoint each run immediately
            with per_run_path.open('w') as f:
                json.dump(detail, f, indent=2)
            print('Saved:', per_run_path)

ablation_df = pd.DataFrame(ablation_rows)
ablation_df

Ablation model: gemma-3-12b-it
Ablation retrievers: ['minilm', 'bge-base']
Ablation epsilons: [10.0, 20.0, 50.0, 100.0]
Ablation lambdas: [0.7, 0.8]
Ablation query count: 250 of 423
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_10_lambda_0p7.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_10_lambda_0p8.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_20_lambda_0p7.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_20_lambda_0p8.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_50_lambda_0p7.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_50_lambda_0p8.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/ablation_results/ablation_results_minilm_epsilon_100_lambda_

PAS defense eval:   0%|          | 0/250 [00:00<?, ?query/s]/Users/shayan/Projects/PAS_RAG/.venv_pas_rag/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


[PAS] Semantic embedder initialized: model='sentence-transformers/all-MiniLM-L6-v2', device='mps'


PAS defense eval: 100%|██████████| 250/250 [29:16<00:00,  7.03s/query, ale=399, ndcg=0.499, recall=0.535]



=== PAS DEBUG: showing 5 low-F1 cases (threshold <= 0.4) ===

--- Failure Case 1 | qid=Q_SYN_1099 | f1=0.0488 | recall=0.0000 | ndcg=0.0000 | ale_m=401.89 ---
Query:
Find libraries w of Prospect Park Grand Army Plaza within 2 miles
Model response:
The provided documents do not list any libraries. They list cafes and museums.
Ground truth (reference answer used for F1):
For query 'Find libraries w of Prospect Park Grand Army Plaza within 2 miles', relevant places include Community Research Library. Community Research Library is categorized as library / research.
Top doc ids:
['DOC_CTX_P82', 'DOC_CTX_C99', 'DOC_CTX_C31', 'DOC_CTX_M51', 'DOC_CTX_M6']
Ground-truth doc ids:
['DOC_CTX_L38']

--- Failure Case 2 | qid=Q_SYN_1276 | f1=0.1039 | recall=0.0000 | ndcg=0.0000 | ale_m=401.89 ---
Query:
Find bars near Metropolitan Museum of Art within 2 miles
Model response:
The provided context does not list any bars. It lists several museums including Heritage History Museum, Upper Brook Art Center

...

PAS defense eval: 100%|██████████| 250/250 [27:34<00:00,  6.62s/query, ale=399, ndcg=0.453, recall=0.482]



=== PAS DEBUG: showing 5 low-F1 cases (threshold <= 0.4) ===

--- Failure Case 1 | qid=Q_SYN_1099 | f1=0.0488 | recall=0.0000 | ndcg=0.0000 | ale_m=401.89 ---
Query:
Find libraries w of Prospect Park Grand Army Plaza within 2 miles
Model response:
The provided documents do not list any libraries. They list cafes and museums.
Ground truth (reference answer used for F1):
For query 'Find libraries w of Prospect Park Grand Army Plaza within 2 miles', relevant places include Community Research Library. Community Research Library is categorized as library / research.
Top doc ids:
['DOC_CTX_P82', 'DOC_CTX_C99', 'DOC_CTX_C31', 'DOC_CTX_M51', 'DOC_CTX_M6']
Ground-truth doc ids:
['DOC_CTX_L38']

--- Failure Case 2 | qid=Q_SYN_1443 | f1=0.0952 | recall=0.0000 | ndcg=0.0000 | ale_m=401.89 ---
Query:
Find pharmacies with photo kiosk s of 71st Ave & Queens Blvd within 1 mile, and return options that best match the requested filters
Model response:
The provided context does not list any pharmacies. 

PAS defense eval: 100%|██████████| 250/250 [27:30<00:00,  6.60s/query, ale=399, ndcg=0.503, recall=0.543]


=== PAS DEBUG: showing 5 low-F1 cases (threshold <= 0.4) ===

--- Failure Case 1 | qid=Q_SYN_1276 | f1=0.1299 | recall=0.0000 | ndcg=0.0000 | ale_m=401.89 ---
Query:
Find bars near Metropolitan Museum of Art within 2 miles
Model response:
The provided documents do not list any bars. They list museums such as Heritage History Museum, Upper Brook Art Center, Upper City History Museum, and Harbor Discovery Hall.
Ground truth (reference answer used for F1):
For query 'Find bars near Metropolitan Museum of Art within 2 miles', relevant places include Lantern Cocktail Bar, Harbor Cocktail Bar, Juniper Wine Bar. Lantern Cocktail Bar is categorized as bar / cocktail. Harbor Cocktail Bar is categorized as bar / cocktail. Juniper Wine Bar is categorized as bar / wine_bar.
Top doc ids:
['DOC_CTX_M23', 'DOC_CTX_M20', 'DOC_CTX_M46', 'DOC_CTX_M31', 'DOC_CTX_M2']
Ground-truth doc ids:
['DOC_CTX_BA23', 'DOC_CTX_BA14', 'DOC_CTX_BA34', 'DOC_CTX_BA20']

--- Failure Case 2 | qid=Q_SYN_1651 | f1=0.1449 | 

,model_name,retriever,epsilon,lambda_weight,pas_mean_recall_at_k,pas_mean_ndcg_at_k,pas_mean_token_f1,pas_mean_citation_correctness,pas_mean_ale_m
0,gemma-3-12b-it,minilm,10.0,0.7,0.4879,0.4608,None,0.5545,398.68
1,gemma-3-12b-it,minilm,10.0,0.8,0.5355,0.5040,None,0.5527,397.04
2,gemma-3-12b-it,minilm,20.0,0.7,0.4901,0.4589,None,0.5439,399.13
3,gemma-3-12b-it,minilm,20.0,0.8,0.5372,0.4980,None,0.5615,399.85
4,gemma-3-12b-it,minilm,50.0,0.7,0.4891,0.4584,None,0.5476,399.85
5,gemma-3-12b-it,minilm,50.0,0.8,0.5412,0.4975,None,0.5631,399.85
6,gemma-3-12b-it,minilm,100.0,0.7,0.4857,0.4600,None,0.5440,399.85
7,gemma-3-12b-it,minilm,100.0,0.8,0.5304,0.4927,None,0.5592,399.85
8,gemma-3-12b-it,bge-base,10.0,0.7,0.4981,0.4706,None,0.5514,397.04
9,gemma-3-12b-it,bge-base,10.0,0.8,0.5365,0.4985,None,0.5527,398.41


In [ ]:
# Aggregate per-run ablation JSON files into one CSV (exclude pas_mean_token_f1)
from pathlib import Path
import pandas as pd
import json

ABL_DIR = OUTPUT_DIR
files = sorted(ABL_DIR.glob('ablation_results_*_epsilon_*_lambda_*.json'))
print('Found ablation files:', len(files))

rows = []
for fp in files:
    try:
        data = json.loads(fp.read_text(encoding='utf-8'))
    except Exception as e:
        print('Skipping unreadable file:', fp.name, e)
        continue

    cfg = data.get('config', {}) or {}
    s = data.get('pas_summary', {}) or {}

    rows.append({
        'file_name': fp.name,
        'model_name': cfg.get('model_name'),
        'retriever': cfg.get('retriever'),
        'epsilon': cfg.get('epsilon'),
        'lambda_weight': cfg.get('lambda_weight'),
        'max_queries': cfg.get('max_queries'),
        'pas_mean_recall_at_k': s.get('mean_recall_at_k'),
        'pas_mean_ndcg_at_k': s.get('mean_ndcg_at_k'),
        'pas_mean_citation_correctness': s.get('mean_citation_correctness'),
        'pas_mean_ale_m': s.get('mean_ale_m'),
    })

agg_df = pd.DataFrame(rows)
if not agg_df.empty:
    agg_df = agg_df.sort_values(by=['retriever', 'epsilon', 'lambda_weight']).reset_index(drop=True)

out_csv = ABL_DIR / 'ablation_metrics_gemma.csv'
agg_df.to_csv(out_csv, index=False)

print('Saved:', out_csv)
agg_df

## Lambda Ablation (Single Retriever + Generator)

Run PAS defense with fixed generator `gemma-3-12b-it` and retriever `minilm`,
sweeping `epsilon in [1, 2]` and `lambda in [0.2, 0.4, 0.5, 0.6]`.
Each run is saved immediately in `lambda_ablations/`.


In [9]:
from copy import deepcopy
import pandas as pd
import json

LAMBDA_ABL_DIR = BASE / 'lambda_ablations'
LAMBDA_ABL_DIR.mkdir(parents=True, exist_ok=True)

LAMBDA_ABL_MODEL = 'gemma-3-12b-it'
LAMBDA_ABL_RETRIEVER = 'minilm'
LAMBDA_ABL_EPSILONS = [1.0, 2.0]
LAMBDA_ABL_LAMBDAS = [0.2, 0.4, 0.5, 0.6]

LAMBDA_ABL_MAX_QUERIES = 250
lambda_queries = queries[:LAMBDA_ABL_MAX_QUERIES]

print('Lambda ablation model:', LAMBDA_ABL_MODEL)
print('Lambda ablation retriever:', LAMBDA_ABL_RETRIEVER)
print('Lambda ablation epsilons:', LAMBDA_ABL_EPSILONS)
print('Lambda ablation lambdas:', LAMBDA_ABL_LAMBDAS)
print('Lambda ablation query count:', len(lambda_queries), 'of', len(queries))
print('Output dir:', LAMBDA_ABL_DIR)


def _fmt_num(x):
    x = float(x)
    return str(int(x)) if x.is_integer() else str(x).replace('.', 'p')


lambda_ablation_rows = []
lambda_ablation_details = {}

base_cfg = deepcopy(CONFIG)
base_cfg['model_name'] = LAMBDA_ABL_MODEL
base_cfg['retriever'] = LAMBDA_ABL_RETRIEVER
apply_retriever_choice(base_cfg)

for eps in LAMBDA_ABL_EPSILONS:
    for lam in LAMBDA_ABL_LAMBDAS:
        eps_s = _fmt_num(eps)
        lam_s = _fmt_num(lam)

        per_run_name = f"lambda_ablation_{LAMBDA_ABL_RETRIEVER}_epsilon_{eps_s}_lambda_{lam_s}.json"
        per_run_path = LAMBDA_ABL_DIR / per_run_name

        if per_run_path.exists():
            print('Skipping existing:', per_run_path)
            try:
                existing = json.loads(per_run_path.read_text(encoding='utf-8'))
                s = existing.get('pas_summary', {}) or {}
                cfg_existing = existing.get('config', {}) or {}
                lambda_ablation_rows.append({
                    'model_name': cfg_existing.get('model_name', LAMBDA_ABL_MODEL),
                    'retriever': cfg_existing.get('retriever', LAMBDA_ABL_RETRIEVER),
                    'epsilon': cfg_existing.get('epsilon', eps),
                    'lambda_weight': cfg_existing.get('lambda_weight', lam),
                    'pas_mean_recall_at_k': s.get('mean_recall_at_k'),
                    'pas_mean_ndcg_at_k': s.get('mean_ndcg_at_k'),
                    'pas_mean_citation_correctness': s.get('mean_citation_correctness'),
                    'pas_mean_ale_m': s.get('mean_ale_m'),
                })
                tag = f"{LAMBDA_ABL_MODEL.replace('/', '-') }__{LAMBDA_ABL_RETRIEVER}__eps{eps_s}__lam{lam_s}"
                lambda_ablation_details[tag] = existing
            except Exception as e:
                print('Warning: could not parse existing file:', e)
            continue

        cfg = deepcopy(base_cfg)
        cfg['epsilon'] = eps
        cfg['lambda_weight'] = lam

        p_summary, p_rows = evaluate_pas_defense(lambda_queries, chunks, anchors, cfg)

        row = {
            'model_name': LAMBDA_ABL_MODEL,
            'retriever': LAMBDA_ABL_RETRIEVER,
            'epsilon': eps,
            'lambda_weight': lam,
            'pas_mean_recall_at_k': p_summary.get('mean_recall_at_k'),
            'pas_mean_ndcg_at_k': p_summary.get('mean_ndcg_at_k'),
            'pas_mean_citation_correctness': p_summary.get('mean_citation_correctness'),
            'pas_mean_ale_m': p_summary.get('mean_ale_m'),
        }
        lambda_ablation_rows.append(row)

        detail = {
            'config': {
                'model_name': LAMBDA_ABL_MODEL,
                'retriever': LAMBDA_ABL_RETRIEVER,
                'epsilon': eps,
                'lambda_weight': lam,
                'max_queries': LAMBDA_ABL_MAX_QUERIES,
            },
            'pas_summary': p_summary,
            'pas_rows': p_rows,
        }

        tag = f"{LAMBDA_ABL_MODEL.replace('/', '-') }__{LAMBDA_ABL_RETRIEVER}__eps{eps_s}__lam{lam_s}"
        lambda_ablation_details[tag] = detail

        with per_run_path.open('w', encoding='utf-8') as f:
            json.dump(detail, f, indent=2)
        print('Saved:', per_run_path)

lambda_ablation_df = pd.DataFrame(lambda_ablation_rows)
if not lambda_ablation_df.empty:
    lambda_ablation_df = lambda_ablation_df.sort_values(by=['epsilon', 'lambda_weight']).reset_index(drop=True)

summary_csv = LAMBDA_ABL_DIR / 'lambda_ablation_summary_gemma_minilm.csv'
summary_json = LAMBDA_ABL_DIR / 'lambda_ablation_summary_gemma_minilm.json'

lambda_ablation_df.to_csv(summary_csv, index=False)
with summary_json.open('w', encoding='utf-8') as f:
    json.dump({'rows': lambda_ablation_rows, 'details': lambda_ablation_details}, f, indent=2)

print('Saved:', summary_csv)
print('Saved:', summary_json)
lambda_ablation_df


Lambda ablation model: gemma-3-12b-it
Lambda ablation retriever: minilm
Lambda ablation epsilons: [1.0, 2.0]
Lambda ablation lambdas: [0.2, 0.4, 0.5, 0.6]
Lambda ablation query count: 250 of 423
Output dir: /Users/shayan/Projects/PAS_RAG/lambda_ablations
Skipping existing: /Users/shayan/Projects/PAS_RAG/lambda_ablations/lambda_ablation_minilm_epsilon_1_lambda_0p2.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/lambda_ablations/lambda_ablation_minilm_epsilon_1_lambda_0p4.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/lambda_ablations/lambda_ablation_minilm_epsilon_1_lambda_0p5.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/lambda_ablations/lambda_ablation_minilm_epsilon_1_lambda_0p6.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/lambda_ablations/lambda_ablation_minilm_epsilon_2_lambda_0p2.json
Skipping existing: /Users/shayan/Projects/PAS_RAG/lambda_ablations/lambda_ablation_minilm_epsilon_2_lambda_0p4.json
Skipping existing: /Users/shayan/Projects/PAS_RAG

PAS defense eval:  26%|██▌       | 64/250 [06:51<34:33, 11.15s/query]

ERROR OCCURRED: Request timed out.


PAS defense eval:  93%|█████████▎| 232/250 [25:29<04:16, 14.27s/query, ale=388, ndcg=0.448, recall=0.49] 

ERROR OCCURRED: Request timed out.


PAS defense eval:  93%|█████████▎| 233/250 [25:48<04:26, 15.70s/query, ale=388, ndcg=0.448, recall=0.49]

ERROR OCCURRED: Request timed out.


PAS defense eval: 100%|██████████| 250/250 [27:53<00:00,  6.69s/query, ale=388, ndcg=0.448, recall=0.49]


=== PAS DEBUG: showing 5 low-F1 cases (threshold <= 0.4) ===

--- Failure Case 1 | qid=Q_SYN_1648 | f1=0.0392 | recall=0.0000 | ndcg=0.0000 | ale_m=143.99 ---
Query:
Find hotels near Bryant Park within 0.5 miles
Model response:
Based on the retrieved context, the best matching hotels are Heritage Discovery Hall, Midtown Corner Study Café, Corner Academic Books, Neighborhood Library, and Midtown Metro Pharmacy.
Ground truth (reference answer used for F1):
For query 'Find hotels near Bryant Park within 0.5 miles', relevant places include Skyline Business Hotel. Skyline Business Hotel is categorized as hotel / business.
Top doc ids:
['DOC_CTX_M40', 'DOC_CTX_C1', 'DOC_CTX_B48', 'DOC_CTX_L1', 'DOC_CTX_PH28']
Ground-truth doc ids:
['DOC_CTX_H66']

--- Failure Case 2 | qid=Q_SYN_1099 | f1=0.0784 | recall=0.0000 | ndcg=0.0000 | ale_m=401.89 ---
Query:
Find libraries w of Prospect Park Grand Army Plaza within 2 miles
Model response:
The provided documents do not list any libraries. They mentio

,model_name,retriever,epsilon,lambda_weight,pas_mean_recall_at_k,pas_mean_ndcg_at_k,pas_mean_citation_correctness,pas_mean_ale_m
0,gemma-3-12b-it,minilm,1.0,0.2,0.4307,0.3966,0.5229,366.92
1,gemma-3-12b-it,minilm,1.0,0.4,0.4336,0.3903,0.5177,366.32
2,gemma-3-12b-it,minilm,1.0,0.5,0.4559,0.4210,0.5431,381.94
3,gemma-3-12b-it,minilm,1.0,0.6,0.4983,0.4601,0.5720,366.85
4,gemma-3-12b-it,minilm,2.0,0.2,0.4258,0.3966,0.5199,387.33
5,gemma-3-12b-it,minilm,2.0,0.4,0.4366,0.4036,0.5265,383.48
6,gemma-3-12b-it,minilm,2.0,0.5,0.4669,0.4281,0.5534,381.06
7,gemma-3-12b-it,minilm,2.0,0.6,0.4803,0.4413,0.5531,388.82
